# SuperKart - Métricas y visualizaciones de negocio

## Objetivo

Analizar la distribución de las ventas registradas en
SuperKart mediante métricas y visualizaciones que permitan
identificar diferencias entre categorías de productos,
establecimientos, ubicaciones y segmentos de precios.

El análisis utiliza los datos procesados en la etapa
de transformación y enriquecimiento.

Los resultados son descriptivos y buscan apoyar la toma
de decisiones comerciales, sin establecer relaciones
causales ni conclusiones sobre rentabilidad.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

proyecto = Path.cwd()

if proyecto.name == "notebooks":
    proyecto = proyecto.parent
elif proyecto.name != "Proyecto-Integrador-EA2":
    proyecto = proyecto / "Proyecto-Integrador-EA2"

ruta_datos = proyecto / "datos" / "datos_procesados.csv"

df = pd.read_csv(ruta_datos)

print("Dimensiones:", df.shape)
display(df.head())

Dimensiones: (7414, 15)


,Product_Id,Product_Sugar_Content,Product_Type,Store_Id,Store_Size,Store_Location_City_Type,Store_Type,Product_Weight,Product_Allocated_Area,Product_MRP,Store_Establishment_Year,Product_Store_Sales_Total,Sales_Range,Price_Range,Sales_Participation_Pct
0,FD6114,Low Sugar,Frozen Foods,OUT004,Medium,Tier 2,Supermarket Type2,12.66,0.027,117.08,2009.0,2842.40,Medio,Bajo,0.012003
1,FD5075,Regular,Canned,OUT001,High,Tier 2,Supermarket Type1,14.28,0.031,162.08,1987.0,4130.16,Muy alto,Muy alto,0.017440
2,FD8233,Low Sugar,Baking Goods,OUT001,High,Tier 2,Supermarket Type1,12.10,0.112,186.31,1987.0,4132.18,Muy alto,Muy alto,0.017449
3,NC1180,No Sugar,Health and Hygiene,OUT002,Small,Tier 3,Food Mart,9.57,0.010,123.67,1998.0,2279.36,Bajo,Medio,0.009625
4,FD5680,Low Sugar,Snack Foods,OUT004,Medium,Tier 2,Supermarket Type2,12.03,0.053,113.64,2009.0,2629.15,Medio,Bajo,0.011102


In [2]:
PALETA = [
    "#14B8A6",
    "#6366F1",
    "#F59E0B",
    "#38BDF8",
    "#EC4899",
    "#84CC16"
]

px.defaults.template = "plotly_dark"
px.defaults.color_discrete_sequence = PALETA

carpeta_graficas = proyecto / "resultados" / "graficas"
carpeta_graficas.mkdir(parents=True, exist_ok=True)

## Indicadores generales

Se calculan los principales indicadores del conjunto
de datos para establecer una visión general de las
ventas registradas.

La moneda y el periodo de ventas no se encuentran
especificados en el dataset, por lo que los resultados
se expresan en unidades monetarias (u. m.).

In [3]:
col_ventas = "Product_Store_Sales_Total"

indicadores = {
    "Ventas totales": df[col_ventas].sum(),
    "Ventas promedio por observación": df[col_ventas].mean(),
    "Mediana de ventas": df[col_ventas].median(),
    "Venta máxima registrada": df[col_ventas].max(),
    "Venta mínima registrada": df[col_ventas].min(),
    "Productos únicos": df["Product_Id"].nunique(),
    "Establecimientos únicos": df["Store_Id"].nunique(),
    "Categorías de productos": df["Product_Type"].nunique(),
    "Observaciones": len(df)
}

resumen_general = pd.DataFrame(
    indicadores.items(),
    columns=["Indicador", "Valor"]
)

display(resumen_general.round(2))

resumen_general.to_csv(
    proyecto / "resultados" / "indicadores_generales.csv",
    index=False
)

,Indicador,Valor
0,Ventas totales,23681606.33
1,Ventas promedio por observación,3194.17
2,Mediana de ventas,3255.06
3,Venta máxima registrada,5462.86
4,Venta mínima registrada,33.00
5,Productos únicos,7414.00
6,Establecimientos únicos,3.00
7,Categorías de productos,16.00
8,Observaciones,7414.00


## Pregunta de negocio 1: Ventas por categoría de producto

¿Qué categorías de productos concentran el mayor valor
de ventas registradas en SuperKart?

El propósito es identificar las categorías con mayor
participación comercial y generar información que apoye
la revisión del portafolio de productos.

### Métricas

- Ventas totales: suma de las ventas registradas por categoría.
  Unidad: unidades monetarias (u. m.).
- Participación porcentual: ventas de la categoría divididas
  entre las ventas totales del dataset, multiplicadas por 100.
  Unidad: porcentaje (%).
- Ventas promedio: ventas totales de la categoría divididas
  entre su número de observaciones.
  Unidad: unidades monetarias por observación.
- Cantidad de observaciones: número de registros por categoría.
  Unidad: registros.

El periodo de ventas no está especificado en la fuente.

In [4]:
def calcular_metricas(datos, columna):
    resumen = datos.groupby(
        columna, observed=True
    ).agg(
        Ventas_totales=(col_ventas, "sum"),
        Ventas_promedio=(col_ventas, "mean"),
        Observaciones=(col_ventas, "count")
    ).reset_index()

    resumen["Participacion_pct"] = (
        resumen["Ventas_totales"] /
        datos[col_ventas].sum()
    ) * 100

    return resumen.sort_values(
        "Ventas_totales", ascending=False
    ).reset_index(drop=True)


ventas_categoria = calcular_metricas(
    df, "Product_Type"
)

display(ventas_categoria.round(2))

ventas_categoria.to_csv(
    proyecto / "resultados" / "ventas_por_categoria.csv",
    index=False
)

,Product_Type,Ventas_totales,Ventas_promedio,Observaciones,Participacion_pct
0,Fruits and Vegetables,3403395.81,3189.69,1067,14.37
1,Snack Foods,3070486.51,3188.46,963,12.97
2,Frozen Foods,2212372.41,3210.99,689,9.34
3,Dairy,2096103.10,3219.82,651,8.85
4,Household,2040758.53,3223.95,633,8.62
5,Baking Goods,1961077.80,3178.41,617,8.28
6,Canned,1847637.54,3147.59,587,7.80
7,Health and Hygiene,1724568.03,3199.57,539,7.28
8,Meat,1608272.26,3141.16,512,6.79
9,Soft Drinks,1431998.42,3217.97,445,6.05


In [5]:
fig_categoria = px.bar(
    ventas_categoria,
    x="Ventas_totales",
    y="Product_Type",
    orientation="h",
    color="Ventas_totales",
    color_continuous_scale=[
        "#1E3A5F",
        "#14B8A6"
    ],
    text="Participacion_pct",
    title="Ventas registradas por categoría de producto",
    labels={
        "Ventas_totales": "Ventas totales (u. m.)",
        "Product_Type": "Categoría de producto",
        "Participacion_pct": "Participación (%)",
        "Ventas_promedio": "Ventas promedio",
        "Observaciones": "Registros"
    },
    hover_data={
        "Ventas_promedio": ":,.2f",
        "Observaciones": True,
        "Participacion_pct": ":.2f"
    }
)

fig_categoria.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False
)

fig_categoria.update_layout(
    height=650,
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    coloraxis_showscale=False,
    yaxis_autorange="reversed",
    margin=dict(l=30, r=80, t=80, b=50)
)

fig_categoria.show()

fig_categoria.write_html(
    carpeta_graficas / "ventas_por_categoria.html"
)

In [6]:
promedio_categoria = ventas_categoria.sort_values(
    "Ventas_promedio",
    ascending=False
)

fig_promedio = px.bar(
    promedio_categoria,
    x="Ventas_promedio",
    y="Product_Type",
    orientation="h",
    text="Ventas_promedio",
    title="Ventas promedio por categoría de producto",
    labels={
        "Ventas_promedio": "Ventas promedio (u. m. por observación)",
        "Product_Type": "Categoría de producto"
    }
)

fig_promedio.update_traces(
    marker_color="#6366F1",
    texttemplate="%{text:,.2f}",
    textposition="outside",
    cliponaxis=False
)

fig_promedio.update_layout(
    height=650,
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    yaxis_autorange="reversed",
    margin=dict(l=30, r=100, t=80, b=50)
)

fig_promedio.show()

fig_promedio.write_html(
    carpeta_graficas / "ventas_promedio_categoria.html"
)

In [7]:
mayor_total = ventas_categoria.iloc[0]

mayor_promedio = ventas_categoria.loc[
    ventas_categoria["Ventas_promedio"].idxmax()
]

menor_total = ventas_categoria.iloc[-1]

participacion_top3 = (
    ventas_categoria.head(3)["Participacion_pct"].sum()
)

print("Categoría con mayores ventas totales:")
print(mayor_total["Product_Type"])
print(f"Ventas: {mayor_total['Ventas_totales']:,.2f}")
print(f"Participación: {mayor_total['Participacion_pct']:.2f}%")

print("\nCategoría con mayor promedio de ventas:")
print(mayor_promedio["Product_Type"])
print(f"Promedio: {mayor_promedio['Ventas_promedio']:,.2f}")

print("\nCategoría con menores ventas totales:")
print(menor_total["Product_Type"])
print(f"Ventas: {menor_total['Ventas_totales']:,.2f}")

print("\nParticipación de las tres principales categorías:")
print(f"{participacion_top3:.2f}%")

assert np.isclose(
    ventas_categoria["Participacion_pct"].sum(), 100
)

Categoría con mayores ventas totales:
Fruits and Vegetables
Ventas: 3,403,395.81
Participación: 14.37%

Categoría con mayor promedio de ventas:
Starchy Foods
Promedio: 3,320.67

Categoría con menores ventas totales:
Seafood
Ventas: 207,066.56

Participación de las tres principales categorías:
36.68%


### Interpretación de resultados

La categoría Fruits and Vegetables presenta las mayores
ventas registradas, con 3.403.395,81 unidades monetarias,
equivalentes al 14,37 % del total.

Las categorías Fruits and Vegetables, Snack Foods y
Frozen Foods concentran el 36,68 % de las ventas.

Por otra parte, Starchy Foods presenta el mayor promedio
de ventas por observación, con 3.320,67 unidades monetarias.

Un resultado relevante es el comportamiento de Seafood,
que registra las menores ventas acumuladas, pero ocupa
el segundo lugar en ventas promedio. Esta categoría
cuenta únicamente con 63 observaciones.

### Recomendación comercial

Se recomienda revisar la disponibilidad y distribución
de las categorías con mayor participación en las ventas.

También sería conveniente estudiar categorías como
Starchy Foods y Seafood, que presentan promedios elevados
a pesar de contar con menos registros.

### Limitaciones

Las diferencias en ventas acumuladas deben interpretarse
considerando la cantidad de observaciones por categoría.

No se dispone de información sobre costos, inventarios,
unidades vendidas o márgenes de utilidad. Por lo tanto,
estos resultados no permiten determinar la rentabilidad
ni justificar por sí solos una ampliación del portafolio.

## Pregunta de negocio 2: Ventas por establecimiento

¿Qué diferencias se observan en las ventas totales y
promedio entre los establecimientos de SuperKart?

El propósito es comparar los resultados comerciales
registrados en los diferentes establecimientos.

### Métricas

- Ventas totales: suma de las ventas por establecimiento.
  Unidad: unidades monetarias.
- Ventas promedio: ventas totales divididas entre la
  cantidad de observaciones del establecimiento.
  Unidad: unidades monetarias por observación.
- Participación porcentual: ventas del establecimiento
  divididas entre las ventas totales del dataset,
  multiplicadas por 100.
  Unidad: porcentaje.
- Cantidad de observaciones: número de registros
  asociados a cada establecimiento.

El periodo de ventas no está especificado.

In [8]:
ventas_establecimientos = calcular_metricas(
    df, "Store_Id"
)

datos_tiendas = df[
    [
        "Store_Id",
        "Store_Type",
        "Store_Size",
        "Store_Location_City_Type"
    ]
].drop_duplicates()

ventas_establecimientos = ventas_establecimientos.merge(
    datos_tiendas,
    on="Store_Id",
    validate="one_to_one"
)

display(ventas_establecimientos.round(2))

ventas_establecimientos.to_csv(
    proyecto / "resultados" / "ventas_por_establecimiento.csv",
    index=False
)

,Store_Id,Ventas_totales,Ventas_promedio,Observaciones,Participacion_pct,Store_Type,Store_Size,Store_Location_City_Type
0,OUT004,15427583.43,3299.31,4676,65.15,Supermarket Type2,Medium,Tier 2
1,OUT001,6223113.18,3923.78,1586,26.28,Supermarket Type1,High,Tier 2
2,OUT002,2030909.72,1762.94,1152,8.58,Food Mart,Small,Tier 3


In [9]:
from plotly.subplots import make_subplots

fig_tiendas = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Ventas totales",
        "Ventas promedio por observación"
    ],
    horizontal_spacing=0.15
)

fig_tiendas.add_trace(
    go.Bar(
        x=ventas_establecimientos["Store_Type"],
        y=ventas_establecimientos["Ventas_totales"],
        marker_color="#14B8A6",
        text=ventas_establecimientos["Participacion_pct"],
        texttemplate="%{text:.1f}%",
        textposition="outside",
        name="Ventas totales"
    ),
    row=1,
    col=1
)

fig_tiendas.add_trace(
    go.Bar(
        x=ventas_establecimientos["Store_Type"],
        y=ventas_establecimientos["Ventas_promedio"],
        marker_color="#6366F1",
        text=ventas_establecimientos["Ventas_promedio"],
        texttemplate="%{text:,.2f}",
        textposition="outside",
        name="Ventas promedio"
    ),
    row=1,
    col=2
)

fig_tiendas.update_yaxes(
    title_text="Ventas totales (u. m.)",
    row=1,
    col=1
)

fig_tiendas.update_yaxes(
    title_text="Ventas promedio (u. m. por observación)",
    row=1,
    col=2
)

fig_tiendas.update_xaxes(
    title_text="Tipo de establecimiento"
)

fig_tiendas.update_layout(
    title="Comparación comercial de establecimientos",
    template="plotly_dark",
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    height=550,
    showlegend=False,
    margin=dict(l=60, r=40, t=100, b=80)
)

fig_tiendas.show()

fig_tiendas.write_html(
    carpeta_graficas / "comparacion_establecimientos.html",
    include_plotlyjs="cdn"
)

In [11]:
display(
    ventas_establecimientos[
        [
            "Store_Id",
            "Store_Type",
            "Observaciones",
            "Ventas_totales",
            "Ventas_promedio",
            "Participacion_pct"
        ]
    ].round(2)
)

assert np.isclose(
    ventas_establecimientos["Ventas_totales"].sum(),
    df[col_ventas].sum()
)

assert np.isclose(
    ventas_establecimientos["Participacion_pct"].sum(),
    100
)

print("Métricas por establecimiento validadas correctamente")

,Store_Id,Store_Type,Observaciones,Ventas_totales,Ventas_promedio,Participacion_pct
0,OUT004,Supermarket Type2,4676,15427583.43,3299.31,65.15
1,OUT001,Supermarket Type1,1586,6223113.18,3923.78,26.28
2,OUT002,Food Mart,1152,2030909.72,1762.94,8.58


Métricas por establecimiento validadas correctamente


### Interpretación de resultados

Supermarket Type2 concentra el 65,15 % de las ventas
registradas y cuenta con 4676 observaciones, siendo
el establecimiento con mayor representación en el dataset.

Supermarket Type1 presenta el mayor promedio de ventas
por observación, con 3923,78 unidades monetarias.

Por su parte, Food Mart registra los menores valores
tanto en ventas totales como en ventas promedio.

### Recomendación comercial

Se recomienda revisar las características del portafolio
de Supermarket Type1 para identificar posibles oportunidades
de mejora en los demás establecimientos.

También sería conveniente analizar la composición de
productos de Food Mart y complementar el estudio con
información de inventarios, costos y demanda.

### Limitaciones

Cada tipo de establecimiento está representado por una
única tienda. Por esta razón, no es posible atribuir las
diferencias observadas exclusivamente al formato comercial.

Las ventas totales también están relacionadas con la
cantidad de observaciones disponibles por establecimiento.

## Pregunta de negocio 3: Ventas por ubicación

¿Cómo se distribuyen las ventas registradas según el
tipo de ubicación de los establecimientos?

El propósito es identificar la concentración de las
ventas en las categorías de ubicación disponibles.

### Métricas

- Ventas totales: suma de las ventas por tipo de ubicación.
  Unidad: unidades monetarias.
- Participación porcentual: ventas de cada ubicación
  divididas entre las ventas totales, multiplicadas por 100.
  Unidad: porcentaje.
- Ventas promedio: ventas totales divididas entre el
  número de observaciones de cada ubicación.
  Unidad: unidades monetarias por observación.
- Número de establecimientos: cantidad de tiendas
  diferentes presentes en cada tipo de ubicación.

El periodo de ventas no está especificado.

In [12]:
ventas_ubicacion = calcular_metricas(
    df, "Store_Location_City_Type"
)

tiendas_ubicacion = df.groupby(
    "Store_Location_City_Type"
)["Store_Id"].nunique().reset_index(
    name="Establecimientos"
)

ventas_ubicacion = ventas_ubicacion.merge(
    tiendas_ubicacion,
    on="Store_Location_City_Type"
)

display(ventas_ubicacion.round(2))

ventas_ubicacion.to_csv(
    proyecto / "resultados" / "ventas_por_ubicacion.csv",
    index=False
)

,Store_Location_City_Type,Ventas_totales,Ventas_promedio,Observaciones,Participacion_pct,Establecimientos
0,Tier 2,21650696.61,3457.47,6262,91.42,2
1,Tier 3,2030909.72,1762.94,1152,8.58,1


In [13]:
fig_ubicacion = px.pie(
    ventas_ubicacion,
    names="Store_Location_City_Type",
    values="Ventas_totales",
    hole=0.65,
    title="Participación de ventas por tipo de ubicación",
    color="Store_Location_City_Type",
    color_discrete_map={
        "Tier 2": "#14B8A6",
        "Tier 3": "#6366F1"
    },
    hover_data=[
        "Ventas_promedio",
        "Observaciones",
        "Establecimientos"
    ]
)

fig_ubicacion.update_traces(
    textposition="inside",
    textinfo="label+percent",
    marker=dict(
        line=dict(color="#0F172A", width=3)
    )
)

fig_ubicacion.update_layout(
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    height=500,
    annotations=[
        dict(
            text=f"{df[col_ventas].sum()/1e6:.2f} M<br>u. m.",
            x=0.5,
            y=0.5,
            font_size=22,
            showarrow=False
        )
    ],
    legend_title_text="Tipo de ubicación"
)

fig_ubicacion.show()

fig_ubicacion.write_html(
    carpeta_graficas / "ventas_por_ubicacion.html",
    include_plotlyjs="cdn"
)

In [14]:
fig_promedio_ubicacion = px.bar(
    ventas_ubicacion,
    x="Store_Location_City_Type",
    y="Ventas_promedio",
    color="Store_Location_City_Type",
    text="Ventas_promedio",
    title="Ventas promedio por tipo de ubicación",
    color_discrete_map={
        "Tier 2": "#14B8A6",
        "Tier 3": "#6366F1"
    },
    labels={
        "Store_Location_City_Type": "Tipo de ubicación",
        "Ventas_promedio": "Ventas promedio (u. m. por observación)"
    }
)

fig_promedio_ubicacion.update_traces(
    texttemplate="%{text:,.2f}",
    textposition="outside",
    cliponaxis=False
)

fig_promedio_ubicacion.update_layout(
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    height=450,
    showlegend=False
)

fig_promedio_ubicacion.show()

fig_promedio_ubicacion.write_html(
    carpeta_graficas / "ventas_promedio_ubicacion.html",
    include_plotlyjs="cdn"
)

assert np.isclose(
    ventas_ubicacion["Participacion_pct"].sum(), 100
)

### Interpretación de resultados

La ubicación Tier 2 concentra el 91,42 % de las ventas
registradas, con un total de 21.650.696,61 unidades
monetarias y 6262 observaciones.

Tier 2 también presenta un promedio de ventas por
observación superior al de Tier 3, con valores de
3457,47 y 1762,94 unidades monetarias, respectivamente.

### Recomendación comercial

Se recomienda revisar la distribución del portafolio
y las condiciones comerciales de los establecimientos
ubicados en Tier 3 para identificar posibles
oportunidades de mejora.

### Limitaciones

Tier 2 está representado por dos establecimientos,
mientras que Tier 3 cuenta únicamente con uno.

El dataset no proporciona información sobre ciudades
específicas, población, competencia o condiciones
económicas de las ubicaciones.

Por lo tanto, no se puede afirmar que la ubicación
sea la causa de las diferencias en ventas.

## Pregunta de negocio 4: Relación entre precio y ventas

¿Qué relación existe entre el precio registrado de los
productos y su valor de ventas?

El propósito es identificar si los productos con precios
más altos tienden a presentar mayores valores de ventas.

### Métricas

- Correlación de Pearson: covarianza entre el precio y
  las ventas dividida entre el producto de sus
  desviaciones estándar.
- Correlación de Spearman: correlación calculada sobre
  los rangos de ambas variables.

Ambos coeficientes son adimensionales y toman valores
entre -1 y 1.

Se utiliza un gráfico de dispersión para observar
el comportamiento conjunto de las variables.

El análisis es descriptivo y no permite establecer
relaciones de causalidad.

In [15]:
col_precio = "Product_MRP"

pearson = df[col_precio].corr(
    df[col_ventas],
    method="pearson"
)

spearman = df[col_precio].rank().corr(
    df[col_ventas].rank(),
    method="pearson"
)

correlaciones = pd.DataFrame({
    "Método": ["Pearson", "Spearman"],
    "Coeficiente": [pearson, spearman]
})

display(correlaciones.round(4))

correlaciones.to_csv(
    proyecto / "resultados" / "correlacion_precio_ventas.csv",
    index=False
)

,Método,Coeficiente
0,Pearson,0.7698
1,Spearman,0.7854


In [16]:
fig_correlacion = px.scatter(
    df,
    x="Product_MRP",
    y=col_ventas,
    color="Store_Type",
    opacity=0.45,
    title="Relación entre precio y ventas registradas",
    labels={
        "Product_MRP": "Precio del producto (u. m.)",
        col_ventas: "Ventas registradas (u. m.)",
        "Store_Type": "Establecimiento"
    },
    hover_data=[
        "Product_Id",
        "Product_Type",
        "Store_Id"
    ],
    color_discrete_map={
        "Supermarket Type2": "#14B8A6",
        "Supermarket Type1": "#6366F1",
        "Food Mart": "#F59E0B"
    }
)

fig_correlacion.update_traces(
    marker=dict(size=5)
)

fig_correlacion.update_layout(
    height=600,
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    legend_title_text="Establecimiento"
)

fig_correlacion.show()

fig_correlacion.write_html(
    carpeta_graficas / "relacion_precio_ventas.html",
    include_plotlyjs="cdn"
)

In [17]:
pendiente, intercepto = np.polyfit(
    df[col_precio],
    df[col_ventas],
    1
)

x_tendencia = np.linspace(
    df[col_precio].min(),
    df[col_precio].max(),
    100
)

y_tendencia = pendiente * x_tendencia + intercepto

fig_tendencia = go.Figure(fig_correlacion)

fig_tendencia.add_trace(
    go.Scatter(
        x=x_tendencia,
        y=y_tendencia,
        mode="lines",
        name="Tendencia lineal global",
        line=dict(
            color="#F8FAFC",
            width=3,
            dash="dash"
        )
    )
)

fig_tendencia.update_layout(
    title=f"Relación entre precio y ventas | Pearson: {pearson:.3f}"
)

fig_tendencia.show()

fig_tendencia.write_html(
    carpeta_graficas / "tendencia_precio_ventas.html",
    include_plotlyjs="cdn"
)

In [18]:
resultados_correlacion = []

for tienda, grupo in df.groupby("Store_Type"):

    resultados_correlacion.append({
        "Establecimiento": tienda,
        "Observaciones": len(grupo),
        "Pearson": grupo[col_precio].corr(
            grupo[col_ventas]
        ),
        "Spearman": grupo[col_precio].rank().corr(
            grupo[col_ventas].rank()
        )
    })

correlacion_tiendas = pd.DataFrame(
    resultados_correlacion
)

display(correlacion_tiendas.round(4))

correlacion_tiendas.to_csv(
    proyecto / "resultados" / "correlacion_por_establecimiento.csv",
    index=False
)

,Establecimiento,Observaciones,Pearson,Spearman
0,Food Mart,1152,0.1517,0.3203
1,Supermarket Type1,1586,0.8184,0.7690
2,Supermarket Type2,4676,0.5616,0.6204


### Interpretación de resultados

El coeficiente de correlación de Pearson global es
aproximadamente 0,770, lo que indica una relación lineal
positiva fuerte entre el precio registrado de los productos
y el valor de sus ventas.

Sin embargo, al realizar el análisis por establecimiento
se identifican diferencias importantes.

Supermarket Type1 presenta una correlación de Pearson
de 0,8184, seguido de Supermarket Type2 con 0,5616
y Food Mart con 0,1517.

Los resultados de Spearman también muestran diferencias
en la intensidad de la asociación entre las variables.

### Recomendación comercial

Se recomienda analizar las estrategias de precios de
manera diferenciada para cada establecimiento, teniendo
en cuenta las características de su portafolio.

También sería conveniente complementar el análisis
con información sobre cantidades vendidas, promociones,
costos y comportamiento de los consumidores.

### Limitaciones

Una correlación positiva no implica que aumentar los
precios genere mayores ventas.

Las ventas se encuentran expresadas en valores monetarios
y no en cantidades de productos vendidos.

La composición de los establecimientos puede influir
en la correlación global. Por esta razón, los resultados
no permiten determinar la elasticidad de la demanda
ni establecer relaciones causales.

In [19]:
ventas_azucar = calcular_metricas(
    df, "Product_Sugar_Content"
)

display(ventas_azucar.round(2))

ventas_azucar.to_csv(
    proyecto / "resultados" / "ventas_por_azucar.csv",
    index=False
)

,Product_Sugar_Content,Ventas_totales,Ventas_promedio,Observaciones,Participacion_pct
0,Low Sugar,13116502.56,3172.07,4135,55.39
1,Regular,6418244.66,3228.49,1988,27.10
2,No Sugar,4146859.11,3212.13,1291,17.51


In [20]:
fig_azucar = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Participación en ventas totales",
        "Ventas promedio por observación"
    ],
    specs=[[{"type": "domain"}, {"type": "xy"}]]
)

colores_azucar = {
    "Low Sugar": "#14B8A6",
    "Regular": "#6366F1",
    "No Sugar": "#F59E0B"
}

colores = [
    colores_azucar[categoria]
    for categoria in ventas_azucar["Product_Sugar_Content"]
]

fig_azucar.add_trace(
    go.Pie(
        labels=ventas_azucar["Product_Sugar_Content"],
        values=ventas_azucar["Ventas_totales"],
        hole=0.55,
        marker=dict(colors=colores),
        textinfo="label+percent"
    ),
    row=1,
    col=1
)

fig_azucar.add_trace(
    go.Bar(
        x=ventas_azucar["Product_Sugar_Content"],
        y=ventas_azucar["Ventas_promedio"],
        marker_color=colores,
        text=ventas_azucar["Ventas_promedio"],
        texttemplate="%{text:,.2f}",
        textposition="outside",
        cliponaxis=False
    ),
    row=1,
    col=2
)

fig_azucar.update_yaxes(
    title_text="Ventas promedio (u. m. por observación)",
    row=1,
    col=2
)

fig_azucar.update_xaxes(
    title_text="Contenido de azúcar",
    row=1,
    col=2
)

fig_azucar.update_layout(
    title="Comportamiento comercial según contenido de azúcar",
    template="plotly_dark",
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    height=550,
    showlegend=False,
    margin=dict(l=40, r=40, t=100, b=60)
)

fig_azucar.show()

fig_azucar.write_html(
    carpeta_graficas / "ventas_por_azucar.html",
    include_plotlyjs="cdn"
)

In [21]:
assert np.isclose(
    ventas_azucar["Participacion_pct"].sum(), 100
)

assert ventas_azucar["Observaciones"].sum() == len(df)

assert "reg" not in df["Product_Sugar_Content"].unique()

print("Métricas validadas correctamente")

display(ventas_azucar.round(2))

Métricas validadas correctamente


,Product_Sugar_Content,Ventas_totales,Ventas_promedio,Observaciones,Participacion_pct
0,Low Sugar,13116502.56,3172.07,4135,55.39
1,Regular,6418244.66,3228.49,1988,27.10
2,No Sugar,4146859.11,3212.13,1291,17.51


## Pregunta de negocio 6: Ventas por segmentos de precios

¿Qué segmentos de precios concentran las mayores ventas
registradas?

El propósito es comparar el comportamiento comercial
de los productos agrupados en cuatro segmentos de
precios definidos mediante cuartiles.

### Métricas

- Ventas totales: suma de las ventas por segmento.
  Unidad: unidades monetarias.
- Ventas promedio: ventas totales divididas entre
  el número de observaciones del segmento.
  Unidad: unidades monetarias por observación.
- Participación porcentual: ventas del segmento
  divididas entre las ventas totales, multiplicadas
  por 100.
  Unidad: porcentaje.

Los segmentos se construyeron durante la etapa de
enriquecimiento de datos y se mantienen constantes
durante el análisis.

El periodo de ventas no está especificado.

In [22]:
orden_rangos = ["Bajo", "Medio", "Alto", "Muy alto"]

ventas_precio = calcular_metricas(
    df, "Price_Range"
)

ventas_precio["Price_Range"] = pd.Categorical(
    ventas_precio["Price_Range"],
    categories=orden_rangos,
    ordered=True
)

ventas_precio = ventas_precio.sort_values(
    "Price_Range"
).reset_index(drop=True)

display(ventas_precio.round(2))

ventas_precio.to_csv(
    proyecto / "resultados" / "ventas_por_rango_precio.csv",
    index=False
)

,Price_Range,Ventas_totales,Ventas_promedio,Observaciones,Participacion_pct
0,Bajo,4216937.46,2274.51,1854,17.81
1,Medio,5578250.63,3010.39,1853,23.56
2,Alto,6436987.18,3473.82,1853,27.18
3,Muy alto,7449431.06,4018.03,1854,31.46


In [23]:
fig_precios = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Ventas totales por segmento",
        "Ventas promedio por segmento"
    ]
)

fig_precios.add_trace(
    go.Bar(
        x=ventas_precio["Price_Range"],
        y=ventas_precio["Ventas_totales"],
        marker_color="#14B8A6",
        text=ventas_precio["Participacion_pct"],
        texttemplate="%{text:.2f}%",
        textposition="outside"
    ),
    row=1,
    col=1
)

fig_precios.add_trace(
    go.Bar(
        x=ventas_precio["Price_Range"],
        y=ventas_precio["Ventas_promedio"],
        marker_color="#6366F1",
        text=ventas_precio["Ventas_promedio"],
        texttemplate="%{text:,.2f}",
        textposition="outside"
    ),
    row=1,
    col=2
)

fig_precios.update_xaxes(
    title_text="Segmento de precios"
)

fig_precios.update_yaxes(
    title_text="Ventas totales (u. m.)",
    row=1,
    col=1
)

fig_precios.update_yaxes(
    title_text="Ventas promedio (u. m. por observación)",
    row=1,
    col=2
)

fig_precios.update_traces(cliponaxis=False)

fig_precios.update_layout(
    title="Comportamiento comercial por segmentos de precios",
    template="plotly_dark",
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    height=550,
    showlegend=False,
    margin=dict(l=60, r=40, t=100, b=60)
)

fig_precios.show()

fig_precios.write_html(
    carpeta_graficas / "ventas_por_rango_precio.html",
    include_plotlyjs="cdn"
)

In [24]:
cruce_segmentos = pd.crosstab(
    df["Price_Range"],
    df["Sales_Range"],
    normalize="index"
).reindex(
    index=orden_rangos,
    columns=orden_rangos
) * 100

display(cruce_segmentos.round(2))

fig_calor = px.imshow(
    cruce_segmentos,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale=[
        "#1E293B",
        "#0D9488",
        "#5EEAD4"
    ],
    labels={
        "x": "Rango de ventas",
        "y": "Rango de precios",
        "color": "Porcentaje (%)"
    },
    title="Distribución de rangos de ventas por segmento de precios"
)

fig_calor.update_layout(
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    height=500
)

fig_calor.show()

fig_calor.write_html(
    carpeta_graficas / "mapa_calor_precio_ventas.html",
    include_plotlyjs="cdn"
)

cruce_segmentos.to_csv(
    proyecto / "resultados" / "cruce_segmentos.csv"
)

Sales_Range,Bajo,Medio,Alto,Muy alto
Price_Range,,,,
Bajo,72.71,22.33,3.40,1.56
Medio,19.10,49.81,25.96,5.13
Alto,4.53,24.07,46.63,24.77
Muy alto,3.72,3.72,24.00,68.55


### Interpretación de resultados

Los productos clasificados como Low Sugar concentran
el 55,39 % de las ventas registradas, con un total
de 13.116.502,56 unidades monetarias.

Sin embargo, la categoría Regular presenta el mayor
promedio de ventas por observación, con 3228,49
unidades monetarias.

Las diferencias entre los promedios de las tres
clasificaciones son reducidas en comparación con
las diferencias en sus ventas acumuladas.

### Recomendación comercial

Se recomienda mantener un seguimiento del portafolio
de productos Low Sugar debido a su participación
en las ventas registradas.

También se recomienda evaluar la composición del
portafolio antes de realizar cambios en la distribución
de productos según su contenido de azúcar.

### Limitaciones

La mayor participación de Low Sugar coincide con una
mayor cantidad de observaciones en el dataset.

Los resultados no permiten afirmar que los consumidores
prefieran productos bajos en azúcar, ya que no se dispone
de información sobre hábitos de consumo, cantidades
vendidas ni decisiones individuales de compra.

### Interpretación de resultados

El segmento de precios muy altos concentra el 31,46 %
de las ventas registradas, con un total de 7.449.431,06
unidades monetarias.

Este segmento también presenta el mayor promedio de
ventas por observación, con 4018,03 unidades monetarias.

Los segmentos de precios altos y muy altos concentran
en conjunto aproximadamente el 58,64 % de las ventas.

El mapa de calor muestra que el 72,71 % de las
observaciones del segmento de precios bajos pertenece
al rango de ventas bajas.

Por otra parte, el 68,55 % de las observaciones del
segmento de precios muy altos pertenece al rango
de ventas muy altas.

Estos resultados son consistentes con la asociación
positiva identificada en el análisis de correlación.

### Recomendación comercial

Se recomienda analizar la composición del portafolio
por segmentos de precios e identificar los productos
que presentan valores de ventas elevados.

Este análisis puede servir como punto de partida para
evaluar estrategias de precios y distribución comercial.

### Limitaciones

Los segmentos fueron construidos mediante cuartiles
del dataset y no corresponden a clasificaciones
comerciales establecidas externamente.

Los mayores valores monetarios de ventas no implican
necesariamente una mayor cantidad de unidades vendidas
ni una mayor rentabilidad.

No se recomienda aumentar los precios basándose
únicamente en estos resultados.

## Validación integral de resultados

Se realizan comprobaciones sobre las métricas calculadas
para garantizar que las agrupaciones conservan el total
de ventas y la cantidad de observaciones del dataset.

También se verifica que las participaciones porcentuales
sumen el 100 % antes de aplicar redondeos y que los
coeficientes de correlación se encuentren dentro de
sus límites matemáticos.

In [25]:
agrupaciones = {
    "Categorías de productos": ventas_categoria,
    "Establecimientos": ventas_establecimientos,
    "Ubicaciones": ventas_ubicacion,
    "Contenido de azúcar": ventas_azucar,
    "Segmentos de precios": ventas_precio
}

validaciones = []

for nombre, tabla in agrupaciones.items():

    total_correcto = np.isclose(
        tabla["Ventas_totales"].sum(),
        df[col_ventas].sum()
    )

    registros_correctos = (
        tabla["Observaciones"].sum() == len(df)
    )

    participacion_correcta = np.isclose(
        tabla["Participacion_pct"].sum(), 100
    )

    validaciones.append({
        "Análisis": nombre,
        "Ventas consistentes": total_correcto,
        "Registros consistentes": registros_correctos,
        "Participación válida": participacion_correcta
    })

    assert total_correcto
    assert registros_correctos
    assert participacion_correcta

assert -1 <= pearson <= 1
assert -1 <= spearman <= 1

assert np.allclose(
    cruce_segmentos.sum(axis=1), 100
)

validacion_final = pd.DataFrame(validaciones)

display(validacion_final)

validacion_final.to_csv(
    proyecto / "resultados" / "validacion_metricas.csv",
    index=False
)

print("Todas las validaciones fueron exitosas")

display(correlaciones.round(4))

,Análisis,Ventas consistentes,Registros consistentes,Participación válida
0,Categorías de productos,True,True,True
1,Establecimientos,True,True,True
2,Ubicaciones,True,True,True
3,Contenido de azúcar,True,True,True
4,Segmentos de precios,True,True,True


Todas las validaciones fueron exitosas


,Método,Coeficiente
0,Pearson,0.7698
1,Spearman,0.7854


## Conclusiones generales

El análisis descriptivo de SuperKart permitió identificar
diferencias comerciales entre categorías de productos,
establecimientos, ubicaciones y segmentos de precios.

Fruits and Vegetables es la categoría con mayores ventas
acumuladas, mientras que Starchy Foods presenta el mayor
promedio de ventas por observación.

Supermarket Type2 concentra la mayor participación en
ventas, aunque Supermarket Type1 registra el promedio
más alto por observación.

La ubicación Tier 2 concentra el 91,42 % de las ventas,
resultado que debe interpretarse considerando la cantidad
de establecimientos y registros disponibles.

El análisis de correlación evidencia una asociación
positiva entre el precio y el valor de ventas, cuya
intensidad varía entre establecimientos.

Los productos Low Sugar concentran la mayor participación
en ventas, aunque los promedios entre las clasificaciones
de contenido de azúcar son similares.

Los segmentos de precios altos y muy altos concentran
una mayor proporción de las ventas registradas.

En conjunto, estos resultados permiten identificar
aspectos relevantes para la revisión del portafolio
y las estrategias comerciales.

Sin embargo, la ausencia de información sobre costos,
cantidades vendidas, periodos de venta y características
de los consumidores limita el alcance de las conclusiones.

La aplicación de Streamlit permitirá consultar estos
resultados de manera interactiva y facilitará su
interpretación por parte de los usuarios.